# Plotting Final — Economic Series Movement Around May 1995

Goal: inspect **how all economic series move during the same calendar year as the reference event, May 1995**. Each series is shown in a shared event window from January 1995 through December 1995, with May 1995 marked clearly. The method is applied consistently to every CSV in `data/cleaned`.

For each series, the notebook plots the observed values during 1995 and the 2- and 4-observation rolling means, plus a standardized view using that series' full-history mean and standard deviation for context. Series with no observations in 1995 are reported as unavailable for that year's movement view rather than silently substituted. Observation frequency may differ between series, so 2 and 4 rolling observations do not always equal 2 and 4 months.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# ---- Shared observation-year controls ----
TARGET_YEAR = 1995
TARGET_MONTH = 5
ROLLING_WINDOWS = (2, 4)

# Event window: the whole calendar year containing the reference month.
WINDOW_START = pd.Timestamp(TARGET_YEAR, 1, 1)
WINDOW_END = pd.Timestamp(TARGET_YEAR, 12, 31)
TARGET_START = pd.Timestamp(TARGET_YEAR, TARGET_MONTH, 1)
TARGET_END = TARGET_START + pd.offsets.MonthEnd(1)

# Locate data regardless of whether Jupyter starts in repository root or notebook folder.
candidates = [Path('data/cleaned'), Path('../data/cleaned'), Path('../../data/cleaned')]
cleaned_dir = next((p.resolve() for p in candidates if p.exists()), None)
if cleaned_dir is None:
    raise FileNotFoundError('Cannot find data/cleaned. Start Jupyter from repository root or notebook folder.')

csv_paths = sorted(cleaned_dir.glob('*.csv'))
if not csv_paths:
    raise FileNotFoundError(f'No CSV files found in {cleaned_dir}')

print(f'Common observation year: {TARGET_YEAR}')
print(f'Event month highlighted: {TARGET_START:%B %Y}')
print(f'Window shown: {WINDOW_START:%Y-%m-%d} to {WINDOW_END:%Y-%m-%d}')
print(f'Rolling windows for every series: {ROLLING_WINDOWS} observations')
print(f'CSV files discovered: {len(csv_paths)}')

series_by_id = {}
load_issues = []
for csv_path in csv_paths:
    series_id = csv_path.stem
    try:
        raw = pd.read_csv(csv_path)
        if not {'date', 'value'}.issubset(raw.columns):
            load_issues.append((series_id, f"Missing date/value columns: {list(raw.columns)}"))
            continue
        raw['date'] = pd.to_datetime(raw['date'], errors='coerce')
        raw['value'] = pd.to_numeric(raw['value'], errors='coerce')
        s = (raw.dropna(subset=['date', 'value'])
                 .sort_values('date')
                 .drop_duplicates('date', keep='last')
                 .set_index('date')['value'])
        if s.empty:
            load_issues.append((series_id, 'No valid date/value observations'))
            continue
        series_by_id[series_id] = s
    except Exception as exc:
        load_issues.append((series_id, str(exc)))

if not series_by_id:
    raise ValueError('No valid economic series could be loaded.')

print(f'Valid series loaded: {len(series_by_id)}')
if load_issues:
    print(f'Series skipped due to data/schema issues: {len(load_issues)}')
    display(pd.DataFrame(load_issues, columns=['series_id', 'issue']))

# Summary explicitly checks data availability during the target year and month.
summary_rows = []
for series_id, s in sorted(series_by_id.items()):
    year_values = s.loc[(s.index >= WINDOW_START) & (s.index <= WINDOW_END)]
    may_values = s.loc[(s.index >= TARGET_START) & (s.index <= TARGET_END)]
    mu, sigma = s.mean(), s.std(ddof=1)
    summary_rows.append({
        'series_id': series_id,
        'first_date': s.index.min(),
        'last_date': s.index.max(),
        'observations_in_1995': len(year_values),
        'observations_in_may_1995': len(may_values),
        'may_1995_value_first_obs': may_values.iloc[0] if len(may_values) else np.nan,
        '1995_value_first_obs': year_values.iloc[0] if len(year_values) else np.nan,
        '1995_value_last_obs': year_values.iloc[-1] if len(year_values) else np.nan,
        'full_history_mean': mu,
        'full_history_sample_std': sigma
    })

summary = pd.DataFrame(summary_rows).sort_values('series_id').reset_index(drop=True)
display(summary)


## All-series movement during calendar year 1995

Each chart is zoomed to January–December 1995 so you can inspect the movement through that year, rather than looking at the entire multi-decade history. May is highlighted as the shared reference month. The standardized panel provides historical context, while the raw panel preserves the series' original units.

In [ ]:
# Apply the same 1995 event-window view to every economic series.
for series_id, s in sorted(series_by_id.items()):
    year_s = s.loc[(s.index >= WINDOW_START) & (s.index <= WINDOW_END)]
    if year_s.empty:
        print(f'{series_id}: no observations in {TARGET_YEAR}; cannot plot its movement during that year.')
        continue

    full_mean = s.mean()
    full_std = s.std(ddof=1)
    year_z = ((year_s - full_mean) / full_std) if pd.notna(full_std) and full_std != 0 else None
    rolling_2 = year_s.rolling(window=ROLLING_WINDOWS[0], min_periods=1).mean()
    rolling_4 = year_s.rolling(window=ROLLING_WINDOWS[1], min_periods=1).mean()
    may_mask = (year_s.index >= TARGET_START) & (year_s.index <= TARGET_END)

    fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True, constrained_layout=True)
    for ax in axes:
        ax.axvspan(TARGET_START, TARGET_END, color='orange', alpha=0.22,
                   label=f'Reference month: {TARGET_START:%B %Y}')

    if year_z is not None:
        axes[0].plot(year_z.index, year_z.values, marker='o', markersize=3,
                     linewidth=1.2, label='Z-score (full-history baseline)')
        axes[0].axhline(0, color='black', linewidth=1, label='Full-history mean')
        axes[0].axhline(1, linestyle='--', alpha=0.5, label='+/- 1 std')
        axes[0].axhline(-1, linestyle='--', alpha=0.5)
        if may_mask.any():
            axes[0].scatter(year_z.loc[may_mask].index, year_z.loc[may_mask].values,
                            color='darkorange', edgecolor='black', zorder=5, label='May observation(s)')
        axes[0].set_ylabel('Z-score')
        axes[0].legend(fontsize=8, ncol=2)
    else:
        axes[0].text(0.5, 0.5, 'Z-score unavailable: insufficient variation',
                     transform=axes[0].transAxes, ha='center', va='center')
        axes[0].set_ylabel('Z-score')

    axes[0].set_title(f'{series_id} — movement during {TARGET_YEAR} (standardized)')
    axes[0].grid(True, alpha=0.25)

    axes[1].plot(year_s.index, year_s.values, marker='o', markersize=3,
                 linewidth=1.2, label='Observed value')
    axes[1].plot(year_s.index, rolling_2.values, linewidth=1.2,
                 label='Rolling mean (2 observations)')
    axes[1].plot(year_s.index, rolling_4.values, linewidth=1.5,
                 label='Rolling mean (4 observations)')
    if may_mask.any():
        axes[1].scatter(year_s.loc[may_mask].index, year_s.loc[may_mask].values,
                        color='darkorange', edgecolor='black', zorder=5, label='May observation(s)')
    axes[1].set_title(f'Raw {series_id} values during {TARGET_YEAR}')
    axes[1].set_ylabel('Original units')
    axes[1].set_xlabel('Date')
    axes[1].legend(fontsize=8, ncol=2)
    axes[1].grid(True, alpha=0.25)
    plt.show()

    print(f'{series_id}: {len(year_s)} observation(s) in {TARGET_YEAR}; '
          f'{int(may_mask.sum())} observation(s) in May.')
    display(year_s.rename('value').to_frame())


## Interpretation checklist

- Follow each series from the start to the end of 1995 and note whether it rises, falls, turns, or remains relatively stable.
- Compare movements around May 1995 across series using the standardized panel; use the raw panel for each series' original units.
- Confirm whether each series actually has a May observation. Monthly, quarterly, and irregular series may show different numbers of points.
- Rolling means are calculated from observations within the displayed 1995 window; the first few values therefore use fewer observations when `min_periods=1`.
- The standardized baseline uses the full available history, including data after 1995. This is suitable for retrospective visualization, not a no-look-ahead trading test.
- Similar-looking movements do not prove correlation, causality, or mean reversion; those require separate statistical tests.

## Derivatives and spreads: May 1995 to December 1995

This section is appended below the existing notebook cells. It does not modify the prior cells. For each series, it calculates the **first difference** between consecutive observations within May–December 1995, which is a discrete derivative proxy (change per observation), and the **spread** as the difference between the current observation and the series' May 1995 starting value. No integral or cumulative-area calculation is used.

Because series can have different units and reporting frequencies, derivative values are per observation—not necessarily per month. The spread is in the series' original units. The first available observation in May is the baseline; if a series has no May observation, its first observation after the start of the window is not silently treated as a May baseline.

In [ ]:
# New section appended after all existing cells: discrete derivatives and spreads.
# Existing notebook cells above remain unchanged.

DERIVATIVE_START = pd.Timestamp(1995, 5, 1)
DERIVATIVE_END = pd.Timestamp(1995, 12, 31)

derivative_tables = {}
derivative_summary_rows = []

for series_id, s in sorted(series_by_id.items()):
    window = s.loc[(s.index >= DERIVATIVE_START) & (s.index <= DERIVATIVE_END)].copy()
    may = s.loc[(s.index >= DERIVATIVE_START) &
                (s.index <= DERIVATIVE_START + pd.offsets.MonthEnd(1))]

    if window.empty:
        derivative_summary_rows.append({
            'series_id': series_id,
            'status': 'No observations from May through December 1995',
            'baseline_date': pd.NaT,
            'baseline_value': np.nan,
            'observations_in_window': 0
        })
        continue

    if may.empty:
        # Do not fabricate a May baseline if the series has no May observation.
        table = pd.DataFrame({
            'value': window,
            'first_difference_per_observation': window.diff(),
            'spread_from_may_1995_baseline': np.nan
        })
        status = 'No May observation; spread from May baseline unavailable'
        baseline_date, baseline_value = pd.NaT, np.nan
    else:
        baseline_date = may.index[0]
        baseline_value = may.iloc[0]
        table = pd.DataFrame({
            'value': window,
            'first_difference_per_observation': window.diff(),
            'spread_from_may_1995_baseline': window - baseline_value
        })
        status = 'OK'

    derivative_tables[series_id] = table
    derivative_summary_rows.append({
        'series_id': series_id,
        'status': status,
        'baseline_date': baseline_date,
        'baseline_value': baseline_value,
        'observations_in_window': len(window)
    })

derivative_summary = pd.DataFrame(derivative_summary_rows)
print('Coverage and baseline summary — May to December 1995')
display(derivative_summary)

for series_id, table in derivative_tables.items():
    print(f'{series_id} — derivatives and spreads (May–December 1995)')
    display(table.rename_axis('date'))

# Combined long-format table for cross-series filtering/export.
combined_derivatives = (
    pd.concat(derivative_tables, names=['series_id', 'date'])
      .reset_index()
      .sort_values(['series_id', 'date'])
      .reset_index(drop=True)
    if derivative_tables else
    pd.DataFrame(columns=[
        'series_id', 'date', 'value',
        'first_difference_per_observation',
        'spread_from_may_1995_baseline'
    ])
)

print('Combined table for all series')
display(combined_derivatives)
